In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

In [ ]:
df_temperature = pd.read_csv("Source_Data/temperaturdaten.csv", encoding="utf-8", sep=";")
df_fitness =  pd.read_csv("Dataframes/workouts_per_second.csv", encoding="utf-8")
df_session = pd.read_csv("Dataframes/session_records.csv", encoding="utf-8")

In [ ]:
df_session.columns

In [ ]:
df_session.dropna(axis=1, how='all', inplace = True)
df_session.describe(include = 'all')

In [ ]:
# Select only numeric columns to avoid dtype errors
numeric_columns = df_session.select_dtypes(include='number')

# Identify columns with zero standard deviation
zero_variance_columns = numeric_columns.columns[numeric_columns.std() == 0]
df_session.drop(columns=zero_variance_columns, inplace=True)

# Identify columns with only one unique value
single_value_columns = df_session.columns[df_session.nunique() == 1]

# Drop those columns from the DataFrame
df_session.drop(columns=single_value_columns, inplace=True)

df_session.describe(include = 'all')

In [ ]:
df_session.drop(columns=['enhanced_avg_speed','enhanced_max_speed','num_laps','unknown_193'], inplace=True)
df_session.describe(include = 'all')

In [ ]:
df_session.head(15)

In [ ]:
df_session.rename(columns={"timestamp": "end_time"}, inplace=True)

In [ ]:
cols = df_session.columns.tolist()
cols.remove("start_time")
cols.remove("end_time")
cols.insert(1, "start_time")
cols.insert(2, "end_time")

df_session = df_session[cols]

In [ ]:
df_session.sample(5)

In [ ]:
df_session.end_time.dtype

In [ ]:
df_session["start_time"] = pd.to_datetime(df_session["start_time"], utc=True)
df_session["end_time"] = pd.to_datetime(df_session["end_time"], utc=True)

In [ ]:
df_session["start_time_Berlin"] = df_session["start_time"].dt.tz_convert("Europe/Berlin")
df_session["end_time_Berlin"] = df_session["end_time"].dt.tz_convert("Europe/Berlin")

cols = df_session.columns.tolist()
cols.remove("start_time_Berlin")
cols.remove("end_time_Berlin")
cols.insert(3, "start_time_Berlin")
cols.insert(4, "end_time_Berlin")

df_session = df_session[cols]                                                                     

In [ ]:
df_session.sample(5)

In [ ]:
df_session[["avg_heart_rate", "max_heart_rate"]] = df_session[["avg_heart_rate", "max_heart_rate"]].replace(0, np.nan)

In [ ]:
df_session[df_session.start_time.isna()].describe()

Die Fitnessdateien enthalten für jede Sekunde eines Workouts einen Datenpunkt. Dabei werden zahlreiche Messwerte in unterschiedlichen Spalten gespeichert. Einige dieser Spalten werden von Zwift jedoch nicht befüllt und enthalten daher keine relevanten Daten.

Im Folgenden wird zunächst ein kurzer Überblick über die vorhandenen Spalten gegeben.


In [ ]:
print(df_fitness.describe())

In [ ]:
print(df_fitness.count() == 0)

In [ ]:
df_fitness.dropna(axis=1, how='all', inplace = True)
print(df_fitness.head())

In [ ]:
print(df_fitness[df_fitness['altitude'] != df_fitness['enhanced_altitude']])
print(df_fitness['speed_1s'].unique())

In [ ]:
df_fitness.drop(columns = ['enhanced_altitude', 'speed_1s'], inplace = True)
print(df_fitness.head())

In [ ]:
# Get source_file values from df_session where start_time is NaT
missing_start_time_files = df_session.loc[df_session['start_time'].isna(), 'source_file']

# Filter df_fitness for rows matching those source_file values
matching_rows = df_fitness[df_fitness['source_file'].isin(missing_start_time_files)]

print(matching_rows)

Es gibt also keine Einträge für Fahrten ohne Startzeit. Ich vermute, dass es sich um Fahrten handelt, bei denen ich an den Start eines Rennens gesprungen bin und es wieder verlassen habe, bevor es startet. Das kommt tatsächlich häufiger vor. Wir können daher alle Fahrten ohne Startzeit löschen.

In [ ]:
df_session.dropna(subset=["start_time"], inplace=True)
df_session.describe(include='all')

In [ ]:
fig = px.histogram(df_session[df_session.total_elapsed_time <= 600], x='total_elapsed_time', nbins=12)
fig.show()

In [ ]:
# Extract date part for grouping
df_session['start_date'] = df_session['start_time'].dt.date

# Count sessions per day
sessions_per_day = df_session.groupby('start_date')['start_date'].transform('count')

# Filter: total_elapsed_time <= 100 AND more than one session on that day
result = df_session[
    (df_session['total_elapsed_time'] <= 100) &
    (sessions_per_day > 1)
]

print(result)

In [ ]:
# Identify rows to delete in df_session
rows_to_delete = df_session[df_session['total_elapsed_time'] <= 100]

# Get the source_file values of the rows to be deleted
files_to_delete = rows_to_delete['source_file']

# Remove corresponding rows in df_fitness
df_fitness = df_fitness[~df_fitness['source_file'].isin(files_to_delete)]

# Remove the rows from df_session
df_session = df_session[df_session['total_elapsed_time'] > 100]

In [ ]:
df_session.describe(include='all')

Der DataFrame df_temperature enthält Messdaten der Station München-Stadt für den Zeitraum vom 10. Februar 2025 bis zum 13. August 2025.

Jeder Datenpunkt umfasst die innerhalb der letzten zehn Minuten gemessene Höchst- und Tiefsttemperatur. Die Messungen wurden in einer Höhe von 2 Metern sowie 5 Zentimetern über dem Boden durchgeführt.

Für die weitere Analyse wird ausschließlich die Lufttemperatur in 2 Metern Höhe verwendet, da diese die Temperatur in meiner Wohnung vermutlich stärker beeinflusst. Die entsprechenden Werte sind in den Spalten TX_10 (Höchsttemperatur) und TN_10 (Tiefsttemperatur) enthalten.

In [ ]:
df_temperature["TM_10"] = (df_temperature["TX_10"] + df_temperature["TN_10"])/2
print(df_temperature)

In [ ]:
df_temperature.drop(columns=['TX5_10', 'TN5_10', 'eor'], inplace=True)
print(df_temperature)

In [ ]:
df_fitness.info()

In [ ]:
df_temperature.info()

QN scheint Leerzeichen zu enthalten, was die Auswertung erschwert. Deshalb entfernen wir alle möglichen Leerzeichen in den column-Namen.

In [ ]:
df_temperature.columns = df_temperature.columns.str.strip()
df_temperature.columns

In [ ]:
# 1. Datumsformate in echte datetime-Objekte umwandeln
df_temperature['MESS_DATUM'] = pd.to_datetime(df_temperature['MESS_DATUM'], format='%Y%m%d%H%M', utc=True)
df_fitness['timestamp'] = pd.to_datetime(df_fitness['timestamp'], format='%Y-%m-%d %H:%M:%S', utc=True)

# 2. Beide DataFrames nach der Zeit sortieren (Voraussetzung für merge_asof)
df_temperature = df_temperature.sort_values('MESS_DATUM')
df_fitness = df_fitness.sort_values('timestamp')

# 3. merge_asof: findet für jeden fitness-Zeitpunkt den passenden
#    MESS_DATUM-Wert, der <= timestamp ist (backward), 
#    aber maximal 10 Minuten zurückliegt (tolerance)
df_merged = pd.merge_asof(
    df_fitness,
    df_temperature[['MESS_DATUM', 'QN', 'TM_10']],
    left_on='timestamp',
    right_on='MESS_DATUM',
    direction='forward',
    tolerance=pd.Timedelta('10min')
)

In [ ]:
df_merged.sample(5)

In [ ]:
df_merged.describe(include='all')

Es gibt noch Temperaturwerte, die -999 angeben. Das ist in dem Datensatz der Defaultwert für nicht vorhanden. Wir schauen uns an, wie viele und wann diese auftauchen.

In [ ]:
df_merged[df_merged.TM_10 == -999]

Das ist nur eine Fahrt. Wir setzen die Temperaturwerte für diese Fahrt auf NaN. Gleichzeitig gab es auch einige fahrten, bei denen ich keinen Herzfrequenzmesser getragen habe. Auch hier werden die Default-0-Werte auf NaN gesetzt. 
In target_power steht ein Wert, wenn ich ein Workout gemacht habe, bei dem man bestimmte Leistungswerte über ein Zeitintervall halten soll. Auch hier werden die Default-0-Werte auf NaN gesetzt. 

In [ ]:
df_merged["heart_rate"] = df_merged["heart_rate"].replace(0, np.nan)
df_merged["TM_10"] = df_merged["TM_10"].replace(-999, np.nan)
df_merged["target_power"] = df_merged["target_power"].replace(0, np.nan)

In [ ]:
df_merged.describe()

Für die Tageszeitauswertung wollen wir auch einen timestamp für die lokale Zeit.

In [ ]:
df_merged['timestamp_berlin'] = df_merged['timestamp'].dt.tz_convert('Europe/Berlin')

Für day_of_week nehme ich aber den UTC-timestamp, da ich auch Fahrten, die kurz nach Mitternacht beginnen noch zum Vortag zählen will.

In [ ]:
df_merged["day_of_week"] = df_merged.timestamp.dt.day_name()
df_merged.sample(5)

In [ ]:
weekdays = list(calendar.day_name)

fig = px.histogram(
    df_merged,
    x="day_of_week",
    title="Number of Rides by day of the week",
    category_orders={"day_of_week": weekdays},
)

fig.update_xaxes(title="day of the week")
fig.update_yaxes(title="seconds")
fig.show()

In [ ]:
df_merged['hour_utc'] = df_merged['timestamp'].dt.hour

fig = px.histogram(df_merged, x='hour_utc', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
df_merged['hour_berlin'] = df_merged['timestamp_berlin'].dt.hour

fig = px.histogram(df_merged, x='hour_berlin', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
def plot_ride_intervals(df, start_col="start_time_Berlin", end_col="end_time_Berlin"):
    """Plot ride intervals with day on the y-axis and time-of-day on the x-axis.

    Only days with at least one ride are shown. Start and end times are
    projected onto a common reference date so the x-axis represents
    time-of-day only (no date component).

    Args:
        df: DataFrame containing ride start and end timestamps.
        start_col: Column name with ride start times (datetime).
        end_col: Column name with ride end times (datetime).
    """
    df_plot = df.copy()

    # Day label used for the y-axis (categorical, only existing days appear)
    df_plot["day"] = df_plot[start_col].dt.strftime("%Y-%m-%d")

    # Project start/end times onto a fixed reference date to get a pure
    # time-of-day axis while keeping them as datetime objects for Plotly
    reference_date = pd.Timestamp("2000-01-01")

    def to_time_of_day(series):
        return reference_date + pd.to_timedelta(
            series.dt.hour * 3600
            + series.dt.minute * 60
            + series.dt.second,
            unit="s",
        )

    df_plot["start_tod"] = to_time_of_day(df_plot[start_col])
    df_plot["end_tod"] = to_time_of_day(df_plot[end_col])

    # Handle rides crossing midnight: end time smaller than start time
    # means it wraps into the next reference day
    crosses_midnight = df_plot["end_tod"] < df_plot["start_tod"]
    df_plot.loc[crosses_midnight, "end_tod"] += pd.Timedelta(days=1)

    df_plot = df_plot.sort_values("day")

    num_days = df_plot["day"].nunique()
    # Scale figure height with the number of days, with a sensible minimum
    fig_height = max(600, num_days * 6)

    fig = px.timeline(
        df_plot,
        x_start="start_tod",
        x_end="end_tod",
        y="day",
        title="Ride Intervals by Day (Time of Day)",
    )

    fig.update_layout(height=fig_height)

    fig.update_yaxes(
        autorange="reversed",
        title="Day",
        categoryorder="array",
        categoryarray=sorted(df_plot["day"].unique(), reverse=True),
    )
    fig.update_xaxes(title="Time of day", tickformat="%H:%M")
    fig.show()

In [ ]:
plot_ride_intervals(df_session)

In [ ]:
# Beispiel-Daten (ersetze 'timestamp' mit deiner Datums-/Zeit-Spalte)
df_merged['time_of_day'] = df_merged['timestamp_berlin'].dt.hour.apply(
    lambda hour:
        'Morning' if 6 <= hour < 12 else
        'Afternoon' if 12 <= hour < 18 else
        'Evening' if 18 <= hour < 24 else
        'Night'
)

In [ ]:
category_order = ["Morning", "Afternoon", "Evening", "Night"]

fig = px.histogram(
    df_merged,
    x="time_of_day",
    title="Number of Rides by Time of Day Category",
    category_orders={"time_of_day": category_order},
)

fig.update_xaxes(title="Time of day")
fig.update_yaxes(title="Number of rides")
fig.show()

In [ ]:
summary = df_merged.groupby("source_file")["heart_rate"].agg(
    total="size",
    nan_count=lambda x: x.isna().sum()
)
summary["nan_ratio"] = summary["nan_count"] / summary["total"]

In [ ]:
print(summary)

In [ ]:
summary[(summary.nan_ratio > 0) & (summary.nan_ratio < 1)]

In [ ]:
df_merged[(df_merged.source_file == "2026-02-15-14-57-18.fit") & (df_merged.heart_rate.isna())]

Das ist ein Zeitintervall ohne Unterbrechung. Schauen wir es uns mit der Zeile davor und danach an.

In [ ]:
df_merged.loc[95634:95643]

Es wurde nicht in die Pedale getreten (power=0). Daher können wir davon ausgehen, dass es in dem Zeitraum keine Spikes gab und wir nähern mit dem Mittelwert der beiden Grenzwerte an.

In [ ]:
mask = (df_merged.source_file == "2026-02-15-14-57-18.fit") & (df_merged.heart_rate.isna())
df_merged.loc[mask, "heart_rate"] = 97

In [ ]:
df_merged[(df_merged.source_file == "2026-06-03-15-30-13.fit") & (df_merged.heart_rate.isna())]

In [ ]:
df_merged.loc[384927:384935]

In [ ]:
subset = df_merged.loc[384900:384950]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=subset.index,
    y=subset["power"],
    name="Power",
    mode="lines+markers",
    yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=subset.index,
    y=subset["heart_rate"],
    name="Heart Rate",
    mode="lines+markers",
    yaxis="y2"
))

fig.update_layout(
    title="Power und Heart Rate",
    xaxis_title="Index",
    yaxis=dict(title="Power (W)", side="left"),
    yaxis2=dict(title="Heart Rate (bpm)", side="right", overlaying="y"),
    legend=dict(x=0.01, y=1.1, orientation="h")
)

fig.show()

Da die getretene Power vernachlässigbar gering ist, nähere ich hier linear an.

In [ ]:
def interpolate_range(df: pd.DataFrame, column: str, start_idx: int, end_idx: int) -> pd.DataFrame:
    """Linearly interpolate NaN values in a column between two indices (inclusive) and round to int.

    Args:
        df: DataFrame containing the column to interpolate.
        column: Name of the column to interpolate.
        start_idx: Start index of the range (inclusive).
        end_idx: End index of the range (inclusive).

    Returns:
        The modified DataFrame (interpolation applied in place).
    """
    mask = (df.index >= start_idx) & (df.index <= end_idx)
    df.loc[mask, column] = df.loc[mask, column].interpolate(method="linear").round()
    return df

In [ ]:
df_merged = interpolate_range(df_merged, "heart_rate", 384927, 384935)

In [ ]:
df_merged.loc[384927:384935]

In [ ]:
df_merged[(df_merged.source_file == "2026-07-11-14-54-53.fit") & (df_merged.heart_rate.isna())]

In [ ]:
df_merged.loc[458967:458979]

In [ ]:
subset = df_merged.loc[458900:458999]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=subset.index,
    y=subset["power"],
    name="Power",
    mode="lines+markers",
    yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=subset.index,
    y=subset["heart_rate"],
    name="Heart Rate",
    mode="lines+markers",
    yaxis="y2"
))

fig.update_layout(
    title="Power und Heart Rate",
    xaxis_title="Index",
    yaxis=dict(title="Power (W)", side="left"),
    yaxis2=dict(title="Heart Rate (bpm)", side="right", overlaying="y"),
    legend=dict(x=0.01, y=1.1, orientation="h")
)

fig.show()

Hier kann ich in den Daten keine Erklärung für den Herzfrequenz-Spike nach den fehlenden Werten entdecken. Es könnte sein, dass irgendetwas passiert ist, was die Daten nicht abbilden (z.B. ich bin schnell abgestiegen und habe etwas geholt, einen Energieriegel). Ich würde die Daten daher trotzdem linear annähern.

In [ ]:
df_merged = interpolate_range(df_merged, "heart_rate", 458967, 458979)

In [ ]:
df_merged[(df_merged.source_file == "2026-02-19-14-18-09.fit") & (df_merged.heart_rate.isna())]

In [ ]:
df_merged.loc[111231:111260]

In [ ]:
df_merged = interpolate_range(df_merged, "heart_rate", 111231, 111260)